In [1]:
pip install pandas numpy faker

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import sqlite3

# Initialize Faker with Indian names to make it realistic for a ZS context
fake = Faker('en_IN')
np.random.seed(42) # This ensures your "random" data stays the same every time you run it

In [3]:
territories = pd.DataFrame({
    'territory_id': range(1, 81),
    'zone': np.random.choice(['North', 'South', 'East', 'West'], 80),
    'city_tier': np.random.choice([1, 2, 3], 80, p=[0.25, 0.45, 0.30]),
    'total_doctors': np.random.randint(80, 500, 80),
    'competitor_presence_score': np.random.randint(1, 11, 80),
    'market_potential_index': np.random.normal(50, 15, 80).clip(10, 100).round(1)
})
territories.head()

,territory_id,zone,city_tier,total_doctors,competitor_presence_score,market_potential_index
0,1,East,1,427,3,54.2
1,2,West,2,310,4,62.4
2,3,North,1,269,4,50.2
3,4,East,3,304,3,71.8
4,5,East,2,464,10,46.0


In [13]:
territories.shape

(80, 6)

In [4]:
reps = pd.DataFrame({
    'rep_id': [f'REP{str(i).zfill(4)}' for i in range(1, 501)],
    'rep_name': [fake.name() for _ in range(500)],
    'territory_id': np.random.randint(1, 81, 500),
    'tenure_months': np.random.randint(3, 120, 500),
    'product_specialty': np.random.choice(['Cardiology', 'Diabetology', 'Oncology', 'Neurology', 'General'], 500),
    'monthly_salary': np.random.normal(45000, 10000, 500).clip(25000, 90000).astype(int)
})
reps.head()

,rep_id,rep_name,territory_id,tenure_months,product_specialty,monthly_salary
0,REP0001,Zayyan Ramaswamy,67,62,Cardiology,51489
1,REP0002,Watika Rai,76,32,Neurology,31080
2,REP0003,Falguni Hayer,26,37,Oncology,36616
3,REP0004,Rajeshri Oommen,16,87,General,50206
4,REP0005,Niharika Mukhopadhyay,51,39,Cardiology,53470


In [15]:
reps.shape

(500, 6)

In [5]:
# Connect to SQLite
conn = sqlite3.connect('pharma_sfe.db')

# Save to Database
reps.to_sql('reps', conn, if_exists='replace', index=False)
territories.to_sql('territories', conn, if_exists='replace', index=False)

# Save to CSV for your GitHub folders
reps.to_csv('reps.csv', index=False)
territories.to_csv('territories.csv', index=False)

In [6]:
prescribers = pd.DataFrame({
    'prescriber_id': [f'DOC{str(i).zfill(5)}' for i in range(1, 5001)],
    'territory_id': np.random.randint(1, 81, 5000),
    'specialty': np.random.choice(['Cardiologist', 'Diabetologist', 'GP'], 5000),
    'monthly_rx_volume': np.random.pareto(3, 5000) * 100 # Right-skewed distribution [cite: 187, 193]
})
# Assign Tiers based on volume
prescribers['prescriber_tier'] = pd.qcut(prescribers['monthly_rx_volume'], 3, labels=['C', 'B', 'A'])

prescribers.head()

,prescriber_id,territory_id,specialty,monthly_rx_volume,prescriber_tier
0,DOC00001,15,Diabetologist,48.603906,A
1,DOC00002,32,Diabetologist,15.079971,B
2,DOC00003,75,Cardiologist,48.378084,A
3,DOC00004,20,Cardiologist,105.166447,A
4,DOC00005,15,GP,23.733521,B


In [18]:
prescribers.shape

(5000, 5)

In [7]:
#We will intentionally add duplicates and missing values here to prove my cleaning skills

calls = pd.DataFrame({
    'call_id': range(1, 50001),
    'rep_id': np.random.choice(reps['rep_id'], 50000),
    'prescriber_id': np.random.choice(prescribers['prescriber_id'], 50000),
    'call_date': [fake.date_between(start_date='-1y', end_date='today') for _ in range(50000)],
    'feedback_score': np.random.randint(1, 11, 50000).astype(float)
})

# Inject 8% Missing Values in feedback_score 
calls.loc[calls.sample(frac=0.08).index, 'feedback_score'] = np.nan

# Inject ~200 Duplicates 
duplicates = calls.sample(200)
calls = pd.concat([calls, duplicates], ignore_index=True)

In [8]:
sales = []
for t_id in range(1, 81):
    for month in range(1, 13):
        sales.append({
            'territory_id': t_id,
            'month': month,
            'revenue': np.random.normal(500000, 100000),
            'target_achievement_pct': np.random.uniform(60, 110)
        })
sales = pd.DataFrame(sales)

sales.head()

,territory_id,month,revenue,target_achievement_pct
0,1,1,595821.292239,65.571374
1,1,2,466997.248984,60.996065
2,1,3,697839.366466,89.771101
3,1,4,539300.959921,83.140945
4,1,5,473757.593943,62.395364


In [25]:
sales.shape

(960, 4)

In [9]:
# Create connection to the SQLite file
conn = sqlite3.connect('pharma_sfe.db')

# Save all tables
reps.to_sql('reps', conn, if_exists='replace', index=False)
territories.to_sql('territories', conn, if_exists='replace', index=False)
prescribers.to_sql('prescribers', conn, if_exists='replace', index=False)
calls.to_sql('calls', conn, if_exists='replace', index=False)
sales.to_sql('sales', conn, if_exists='replace', index=False)

# Also save CSVs for your GitHub 'raw data' folder 
reps.to_csv('reps.csv', index=False)
territories.to_csv('territories.csv', index=False)
prescribers.to_csv('prescribers.csv', index=False)
calls.to_csv('calls.csv', index=False)
sales.to_csv('sales.csv', index=False)

conn.close()
print("Database 'pharma_sfe.db' and 5 CSVs created successfully!")

Database 'pharma_sfe.db' and 5 CSVs created successfully!


In [12]:
# Recalculate tiers based on actual volume percentiles
prescribers['prescriber_tier'] = pd.qcut(
    prescribers['monthly_rx_volume'], 
    3, 
    labels=['C', 'B', 'A']
)

In [14]:
# 1. Join calls with reps to get territory_id (needed for median calculation)
calls_merged = calls.merge(reps[['rep_id', 'territory_id']], on='rep_id', how='left')

# 2. Fill missing feedback_score with the median of that territory
calls_merged['feedback_score'] = calls_merged.groupby('territory_id')['feedback_score'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Verify it exists now
print(f"Variable 'calls_merged' created. Shape: {calls_merged.shape}")

Variable 'calls_merged' created. Shape: (50200, 6)


In [15]:
# Connect to a NEW database for processed data
clean_conn = sqlite3.connect('processed_pharma_sfe.db')

# Save cleaned tables
calls_merged.to_sql('calls_cleaned', clean_conn, if_exists='replace', index=False)
prescribers.to_sql('prescribers_cleaned', clean_conn, if_exists='replace', index=False)

clean_conn.close()
print("Cleaning complete! Processed data saved.")

Cleaning complete! Processed data saved.
